# 엑셀/PDF 파일 읽기 실습

- 엑셀 파일은 `openpyxl`, PDF 파일은 `pypdf` 패키지를 사용


## 패키지 설치

터미널에서 아래 명령어를 한 번 실행한다.

```bash
python -m pip install openpyxl pypdf
```


## 엑셀 파일 읽기

엑셀 파일은 단순 문자열 파일이 아니라 여러 시트, 셀, 서식 정보가 들어 있는 구조화 파일이다. 그래서 `openpyxl` 같은 전용 라이브러리로 읽는다.


In [26]:
# load_workbook(): 엑셀을 열어서 Workbook 객체로 변환
from openpyxl import load_workbook

#data_only=True : 셀에 작성된 데이터 읽어오기(수식 -> 수식의 결과 읽어오기)
workbook = load_workbook('students.xlsx', data_only=True)

print(workbook.sheetnames)

worksheet = workbook['students'] # 'students' 시트만 얻어오기
print(worksheet.title)# 시트 이름

print('max row:',worksheet.max_row)
print('max col:',worksheet.max_column)




['students']
students
max row: 5
max col: 5


In [27]:
# 특정 셀의 값 읽어오기
print(worksheet['B2'].value)
print(worksheet['D3'].value)

print('4행 2열',worksheet.cell(row=4, column=2).value)



홍길동
85
4행 2열 유관순


In [28]:
# 엑셀 한줄씩 읽기
# iter == iterator == 반복자
# iter_rows == 행 반복자: 반복될때마다 다음 행 반환
# values_only=True : 셀 안의 실제 값만 tuple로 가져오기
for row in worksheet.iter_rows(values_only=True):
    print(row)

('id', 'name', 'course', 'score', 'passed')
(1, '홍길동', 'Python IO', 92, True)
(2, '이순신', 'Python IO', 85, True)
(3, '유관순', 'Python IO', 78, True)
(4, '신사임당', 'Python IO', 64, False)


In [29]:
# 엑셀 행 데이터를 dict list로 변환
rows = list(worksheet.iter_rows(values_only=True))
print(rows)

headers = rows[0]

student_list : list[dict] = []
for row in rows[1:]:
    #학생 dict 생성
    #zip(tuple, tuple) -> 두튜플의 같은 인덱스 요소끼리 k:v 쌍으로 묶어서 반환
    student = dict(zip(headers, row))
    student_list.append(student)

for student in student_list:
    print(student)

[('id', 'name', 'course', 'score', 'passed'), (1, '홍길동', 'Python IO', 92, True), (2, '이순신', 'Python IO', 85, True), (3, '유관순', 'Python IO', 78, True), (4, '신사임당', 'Python IO', 64, False)]
{'id': 1, 'name': '홍길동', 'course': 'Python IO', 'score': 92, 'passed': True}
{'id': 2, 'name': '이순신', 'course': 'Python IO', 'score': 85, 'passed': True}
{'id': 3, 'name': '유관순', 'course': 'Python IO', 'score': 78, 'passed': True}
{'id': 4, 'name': '신사임당', 'course': 'Python IO', 'score': 64, 'passed': False}


In [30]:
# 엑셀에서 읽어온 학생 데이터로 학생수, 합계, 평균, 최저, 최고 점수 구하기
sum_score = 0
min_score = 100
max_score = 0
for student in student_list:
    score = student['score']
    sum_score += score

    if min_score > score:
        min_score = score

    if max_score < score:
        max_score = score
length = len(student_list)
print(f'''students: {length},
sum: {sum_score},
avg: {sum_score/length},
min: {min_score},
max: {max_score}''')

students: 4,
sum: 319,
avg: 79.75,
min: 64,
max: 92


In [33]:
del sum
del min
del max
#sum, min, max 위에서 변수명으로 썼다가 대참사가 일어남,

score_list = [x['score'] for x in student_list]
# print(type(score_list[0])
print(f'''
students: {len(score_list)},
sum: {sum(score_list)},
avg: {sum(score_list)/len(score_list)},
min: {min(score_list)},
max: {max(score_list)}''')

students: 4,
sum: 319,
avg: 79.75,
min: 64,
max: 92


## PDF 파일 읽기

- PDF는 페이지, 글자 위치, 폰트, 이미지 정보가 함께 들어 있는 문서 파일이다. 그래서 텍스트 파일처럼 바로 읽기보다 `pypdf` 같은 라이브러리로 페이지 단위 텍스트를 추출한다.

- 한글 PDF는 폰트와 인코딩 방식에 따라 텍스트 추출 결과가 달라질 수 있음



In [38]:
# pypdf: 파이썬에서 pdf 파일을 읽고 쓸수 있게 하는 라이브러리(모듈)
from pypdf import PdfReader

# pathlib.Path: 파일/폴더 경로를 편하게 다루기 위한 클래스
# 현재 노트북파이링 실행되는 위치를 기준으로 한다(기본값)
from pathlib import Path

pdf_path = Path('io_sample.pdf')
# print(pdf_path)

reader = PdfReader(pdf_path)
# print(reader, type(reader))

# PdfReader.pages: 현재 읽어들인 pdf 파일의 페이지 객체를 담아둔 list
print('reader.pages', reader.pages)
print('읽은 페이지수: ', len(reader.pages))


incorrect startxref pointer(1)
parsing for Object Streams


reader.pages [PageObject(0), PageObject(1)]
읽은 페이지수:  2


In [44]:
# 첫페이지 텍스트 추출
first_page_text = reader.pages[0].extract_text()
print(first_page_text)


IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.



In [48]:
# 전체페이지 text추출
for idx, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    print(f'''{idx} 번 페이지:
{text}    ''')


1 번 페이지:
IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.
    
2 번 페이지:
Second Page
A PDF can contain multiple pages.
In Python, each page can be read and processed separately.
    
